In [1]:
import pandas as pd
import re
from io import StringIO

# 1. Paste your raw text here
raw_data_text = """
  Year 2020:
            Country  Actual  Predicted   Error
      Venezuela, RB -29.999     -0.632 -29.367
            Uruguay  -7.357     -0.632  -6.725
               Peru -10.933     -0.632 -10.301
          Paraguay  -0.820     -0.632  -0.188
            Panama -17.821     -0.632 -17.189
          Nicaragua  -2.237      1.557  -3.793
            Mexico  -8.354      1.557  -9.911
          Honduras  -8.965      1.557 -10.522
              Haiti  -3.306      1.557  -4.862
          Guatemala  -1.786      1.557  -3.342
        El Salvador  -7.893      1.617  -9.510
            Ecuador  -9.245      1.617 -10.862
Dominican Republic  -7.929      1.617  -9.546
              Cuba -10.949      1.617 -12.566
        Costa Rica  -4.273      1.617  -5.890
          Colombia  -7.186      1.385  -8.571
              Chile  -6.143      1.385  -7.528
            Brazil  -3.277      1.385  -4.662
            Bolivia -12.719      1.385 -14.104
          Argentina  -9.900      1.385 -11.285

  Year 2021:
            Country  Actual  Predicted  Error
      Venezuela, RB   0.955     -9.314 10.270
            Uruguay   5.844     -9.314 15.159
               Peru  13.363     -9.314 22.677
          Paraguay   4.017     -9.314 13.331
            Panama  16.467     -9.314 25.782
          Nicaragua  10.454     -5.469 15.924
            Mexico   6.048     -5.469 11.518
          Honduras  12.565     -5.469 18.034
              Haiti  -1.798     -5.469  3.671
          Guatemala   8.042     -5.469 13.511
        El Salvador  11.905     -7.185 19.090
            Ecuador   9.422     -7.185 16.607
Dominican Republic  14.012     -7.185 21.198
              Cuba   1.254     -7.185  8.440
        Costa Rica   7.936     -7.185 15.121
          Colombia  10.801     -7.120 17.921
              Chile  11.315     -7.120 18.435
            Brazil   4.763     -7.120 11.882
            Bolivia  10.029     -7.120 17.149
          Argentina  10.442     -7.120 17.562

  Year 2022:
            Country  Actual  Predicted  Error
      Venezuela, RB   8.001      6.689  1.312
            Uruguay   4.486      6.689 -2.203
               Peru   2.809      6.689 -3.880
          Paraguay   0.176      6.689 -6.513
            Panama  11.039      6.689  4.350
          Nicaragua   3.550      6.776 -3.226
            Mexico   3.710      6.776 -3.066
          Honduras   4.144      6.776 -2.633
              Haiti  -1.682      6.776 -8.458
          Guatemala   4.185      6.776 -2.592
        El Salvador   2.955      7.118 -4.163
            Ecuador   5.868      7.118 -1.249
Dominican Republic   5.238      7.118 -1.880
              Cuba   1.775      7.118 -5.343
        Costa Rica   4.551      7.118 -2.566
          Colombia   7.328      6.935  0.393
              Chile   2.154      6.935 -4.781
            Brazil   3.017      6.935 -3.918
            Bolivia   3.747      6.935 -3.187
          Argentina   6.021      6.935 -0.914

  Year 2023:
            Country  Actual  Predicted  Error
      Venezuela, RB   4.002      3.207  0.794
            Uruguay   0.742      3.207 -2.465
               Peru  -0.403      3.207 -3.610
          Paraguay   4.999      3.207  1.792
            Panama   7.166      3.207  3.959
          Nicaragua   4.428      2.267  2.161
            Mexico   3.354      2.267  1.086
          Honduras   3.576      2.267  1.309
              Haiti  -1.864      2.267 -4.131
          Guatemala   3.533      2.267  1.266
        El Salvador   3.539      2.695  0.845
            Ecuador   1.988      2.695 -0.706
Dominican Republic   2.192      2.695 -0.502
              Cuba  -1.929      2.695 -4.624
        Costa Rica   5.112      2.695  2.417
          Colombia   0.712      2.736 -2.023
              Chile   0.521      2.736 -2.214
            Brazil   3.242      2.736  0.506
            Bolivia   2.516      2.736 -0.219
          Argentina  -1.856      2.736 -4.591

  Year 2024:
            Country  Actual  Predicted  Error
      Venezuela, RB   5.300      1.436  3.864
            Uruguay   3.108      1.436  1.673
               Peru   3.304      1.436  1.868
          Paraguay   4.250      1.436  2.814
            Panama   2.748      1.436  1.312
          Nicaragua   3.588      1.505  2.083
            Mexico   1.427      1.505 -0.078
          Honduras   3.554      1.505  2.049
              Haiti  -4.170      1.505 -5.675
          Guatemala   3.652      1.505  2.147
        El Salvador   2.602      1.343  1.259
            Ecuador  -2.001      1.343 -3.344
Dominican Republic   4.954      1.343  3.611
              Cuba  -1.062      1.343 -2.405
        Costa Rica   4.321      1.343  2.978
          Colombia   1.598      0.753  0.845
              Chile   2.644      0.753  1.891
            Brazil   3.419      0.753  2.666
            Bolivia  -1.123      0.753 -1.877
          Argentina  -1.343      0.753 -2.096
"""

# 2. Parse the text
parsed_data = []
current_year = None

for line in raw_data_text.split('\n'):
    year_match = re.search(r'Year (\d{4}):', line)
    if year_match:
        current_year = year_match.group(1)
        continue
    
    # Matches: Country Name, Actual, Predicted, Error
    row_match = re.match(r'^\s*(.*?)\s+(-?\d+\.\d+)\s+(-?\d+\.\d+)\s+(-?\d+\.\d+)\s*$', line)
    if row_match and current_year:
        country, actual, predicted, error = row_match.groups()
        parsed_data.append({
            'Year': f"{current_year}-01-01", 
            'Country': country.strip(),
            'Predicted': float(predicted)
        })

df_raw = pd.DataFrame(parsed_data)

# 3. Map Country names to ISO codes
iso_mapping = {
    'Haiti': 'HTI', 'Paraguay': 'PRY', 'Brazil': 'BRA', 'Guatemala': 'GTM',
    'Costa Rica': 'CRI', 'Nicaragua': 'NIC', 'Mexico': 'MEX', 'Uruguay': 'URY',
    'Cuba': 'CUB', 'Chile': 'CHL', 'Colombia': 'COL', 'Ecuador': 'ECU',
    'Bolivia': 'BOL', 'El Salvador': 'SLV', 'Honduras': 'HND', 'Argentina': 'ARG',
    'Peru': 'PER', 'Dominican Republic': 'DOM', 'Panama': 'PAN', 'Venezuela, RB': 'VEN'
}
df_raw['ISO'] = df_raw['Country'].map(iso_mapping)

# 4. Pivot into the final shape (Years as index, ISO codes as columns)
new_fts_forecasts_cv = df_raw.pivot(index='Year', columns='ISO', values='Predicted')
new_fts_forecasts_cv.index = pd.to_datetime(new_fts_forecasts_cv.index)

print("Successfully generated new_fts_forecasts_cv!")

Successfully generated new_fts_forecasts_cv!
